# IRC Labels: Catalog Metadata Enrichment for Iceberg

This notebook demonstrates the [IRC Labels proposal](https://github.com/apache/iceberg/pull/15750) —
a new optional `labels` field in `LoadTableResponse` that lets catalogs enrich table metadata
with operational context: ownership, discovery, cost attribution, and semantic hints.

**Labels are a general-purpose metadata channel** — not just governance.

**Prerequisites**: Run `setup/bootstrap.py` first to create warehouse and tables.

In [ ]:
from pyiceberg.catalog import load_catalog
import json

S3 = {"s3.endpoint": "http://localhost:9000", "s3.access-key-id": "admin",
      "s3.secret-access-key": "password", "s3.region": "us-east-1"}

# Through Labels Proxy (enriches LoadTableResponse with labels)
catalog = load_catalog("healthcare", type="rest",
    uri="http://localhost:8182/catalog", warehouse="healthcare", **S3)

tables = {}
for tid in catalog.list_tables("healthcare"):
    t = catalog.load_table(tid)
    tables[tid[1]] = t
    df = t.scan().to_arrow()
    print(f"Loaded: {tid[1]:20s}  rows={len(df)}  labels={'yes' if t.labels else 'no'}")

---
## Part 1: Labels in the IRC Protocol

The `labels` field is optional in `LoadTableResponse`.
Catalogs populate it; engines read or ignore it. Table metadata files are unchanged.

In [ ]:
patients = tables["patients"]
print("=== table.labels ===")
print(json.dumps(patients.labels, indent=2))

print("\n=== Convenience accessors ===")
print(f"table.table_labels:  {patients.table_labels}")
print(f"table.column_labels: {len(patients.column_labels)} columns annotated")
print(f"get_column_label(3): {patients.get_column_label(3)}")

In [ ]:
# Same table loaded DIRECTLY from Lakekeeper (no proxy) — no labels
direct = load_catalog("direct", type="rest",
    uri="http://localhost:8181/catalog", warehouse="healthcare", **S3)
p_direct = direct.load_table(("healthcare", "patients"))

print(f"Through proxy  -> labels: {bool(patients.labels)} ({len(patients.labels)} keys)")
print(f"Direct catalog -> labels: {bool(p_direct.labels)} (empty)")
print("\n-> Labels are ephemeral catalog enrichment. Same table, different context.")

---
## Part 2: Ownership Tracking

**Who owns this table? Who do I page when it breaks?**
Today this lives in wikis. Labels make it machine-readable.

In [ ]:
print(f"{'Table':<20} {'Owner':<20} {'Tier':<8} {'Domain'}")
print("-" * 60)
for name, t in tables.items():
    l = t.table_labels
    print(f"{name:<20} {l.get('owner_team','?'):<20} {l.get('tier','?'):<8} {l.get('domain','?')}")

In [ ]:
broken = "visits_summary"
owner = tables[broken].table_labels.get("owner_team", "?")
tier = tables[broken].table_labels.get("tier", "?")
print(f"Table '{broken}' is broken.  Owner: {owner}, Tier: {tier}")
if tier == "gold":
    print(f"  -> Page {owner} immediately (gold = P1 SLA)")
else:
    print(f"  -> Ticket for {owner} (standard SLA)")

---
## Part 3: Discovery — Self-Documenting Tables

Labels carry domain, tier, description, quality scores — 80% of a data catalog for free.

In [ ]:
# Data dictionary from column labels
table = tables["patients"]
print(f"Data Dictionary: healthcare.patients")
print(f"Description: {table.table_labels.get('description', 'N/A')}\n")
print(f"{'Column':<15} {'Type':<12} {'Meaning':<40} {'Semantic Type'}")
print("-" * 90)
for field in table.schema().fields:
    cl = table.get_column_label(field.field_id)
    meaning = cl.get('meaning', '--')
    sem = cl.get('semantic_type', cl.get('pii_type', cl.get('phi_type', '--')))
    print(f"{field.name:<15} {str(field.field_type):<12} {meaning:<40} {sem}")

In [ ]:
# All tables with data preview
for name, t in tables.items():
    l = t.table_labels
    df = t.scan().to_arrow().to_pandas()
    print(f"\n{'='*70}")
    print(f"{name}  |  sensitivity={l.get('sensitivity','?')}  |  owner={l.get('owner_team','?')}  |  {len(df)} rows")
    print(f"{'='*70}")
    print(df.head(5).to_string(index=False))

---
## Part 4: FinOps — Cost Attribution

Labels carry ephemeral operational context. No table commits needed.

In [ ]:
print(f"{'Table':<20} {'Cost Center':<25} {'Tier':<8} {'Domain'}")
print("-" * 65)
for name, t in tables.items():
    l = t.table_labels
    print(f"{name:<20} {l.get('cost_center','unattributed'):<25} {l.get('tier','?'):<8} {l.get('domain','?')}")

print("\nSLA Config:")
sla = {"gold": "P1 (15min)", "silver": "P2 (1hr)", "bronze": "P3 (4hr)"}
for name, t in tables.items():
    tier = t.table_labels.get('tier', 'bronze')
    print(f"  {name}: {sla.get(tier, 'P3')}")

---
## Takeaways

Labels are a **general-purpose metadata channel** in IRC.

| Use Case | Labels | What it enables |
|----------|--------|----------------|
| Ownership | `owner_team`, `tier` | Incident routing, on-call |
| Discovery | `domain`, `description` | Self-documenting tables |
| FinOps | `cost_center`, `sla_tier` | Attribution without commits |

**See also:** [governance.ipynb](governance.ipynb) — governance via trusted engine (ClickHouse)